# Handwritten Tabular Form — Pre-processing Pipeline (R&D)

Turns the scanned handwritten forms (**Supplier Lot Rejection, Supplier Rejection, Supplier Rework, Supplier Segregation,
Weld Shop Rejection**) into clean, **per-column cell crops that are ready for OCR**. *(Stages 0–13 stop at the crops; Stages 14–16 then run GOT-OCR-2.0 on every cell and save the readings
to one JSON file.)*

Rows and columns come from the *printed grid* found with OpenCV. Every crop keeps its real pixel position, so the
row/column relationship is preserved by geometry alone and each crop is filed under its real **column header name**.

| # | Stage | What it does |
|---|-------|--------------|
| 0 | Setup | installs / imports libraries |
| 1 | Config | DPI, padding, paths, debug switches |
| 2 | Templates | column (header) names per form |
| 3 | Manual overrides | column X / row Y coordinates if auto-detection fails |
| 4 | Input | Google Drive folder, file upload, or local folder |
| 5 | PDF → image | every page rendered at **300 DPI** (PyMuPDF, `pdf2image` fallback) |
| 6 | Pre-processing | grayscale → denoise → illumination flatten → **deskew / de-shear** → CLAHE contrast |
| 7 | Table detection | morphological line masks → table bounding box |
| 8 | Rows / columns | grid line positions from projection profiles |
| 9 | Grid resolution | map lines to header names, apply manual overrides |
| 10 | Records | group printed grid rows into handwritten records (text wraps onto a 2nd printed row) |
| 11 | Crops | per-record, per-column crops with **padding**; last column extends to the right |
| 12 | Line removal | erase the printed grid lines from each crop |
| 13 | Debug views | original, table, rows/columns, crop boxes, crops, line-removed crops |
| 14 | OCR model | loads `GOT-OCR-2.0` (pretrained, no fine-tuning) |
| 15 | Cell reader | ink-trim each crop; GOT reads the whole cell as one image |
| 16 | OCR + JSON | every non-empty cell through GOT-OCR-2.0 -> `ocr_output/ocr_cell_results.json` |
| 17 | Structured CSV | pivot that JSON back into the shape of the form: **one CSV per PDF**, one row per record, one column per header |
| 18 | Save + download | `ocr_df` -> CSV, zip the output folder, download it as one zip (Colab) |

> Output: `ocr_output/debug/<page>/` (all intermediate images) and `ocr_output/crops/<page>/` (raw + line-removed crop per cell,
> named `r01_<COLUMN NAME>.png`).
> `ocr_output/ocr_cell_results.json` holds the GOT text + confidence for every cell (page → record → column);
> `ocr_output/structured/<pdf>.csv` is the same data as a table — one file per source PDF, one row per record,
> one column per form header.


In [ ]:
# ============================================================================
# STAGE 0 — SETUP
# Installs only what is missing (Colab already ships torch, transformers, OpenCV,
# pandas and matplotlib; PyMuPDF is the one usually needed).
# ============================================================================
import importlib.util, subprocess, sys

def ensure(pip_name, module=None):
    """pip-install `pip_name` if `module` cannot be imported."""
    if importlib.util.find_spec(module or pip_name) is None:
        print(f"installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

for pip_name, module in [("pymupdf", "pymupdf"), ("opencv-python-headless", "cv2"),
                         ("transformers", "transformers"), ("torch", "torch"),
                         ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("pillow", "PIL")]:
    ensure(pip_name, module)
print("setup OK")


installing pymupdf ...
setup OK


In [ ]:
# ============================================================================
# STAGE 1 — CONFIG + SMALL UTILITIES
# Everything you may want to tune lives here.
# ============================================================================
import os, re, json, glob, math, time, shutil, warnings
from dataclasses import dataclass, field
from datetime import datetime
from typing import Optional

import numpy as np
import cv2
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont

warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules

# ---- permanent storage (Colab) -------------------------------------------------
# Colab's own disk (/content) is wiped when the runtime ends, so on Colab the uploads AND every
# output file live in this Google Drive folder instead - they are still there after a restart.
DRIVE_ROOT = "/content/drive/MyDrive/MeenakshiPublic"

def mount_drive():
    """Mount Google Drive once (Colab only). Safe to call again - it is a no-op when already mounted."""
    if not IN_COLAB:
        return
    from google.colab import drive                      # only available inside Colab
    if not os.path.isdir("/content/drive/MyDrive"):
        drive.mount("/content/drive")

mount_drive()                                           # needed before the makedirs further down

# ---- where the PDFs come from -------------------------------------------------
INPUT_MODE     = "drive" if IN_COLAB else "local"      # "drive" | "upload" | "local"
DRIVE_PDF_DIR  = f"{DRIVE_ROOT}/Datasetpdf"             # used when INPUT_MODE == "drive"
LOCAL_PDF_DIR  = "Dataset"                              # used when INPUT_MODE == "local"
UPLOAD_DIR     = f"{DRIVE_ROOT}/uploads" if IN_COLAB else "uploads"      # used when INPUT_MODE == "upload"
ONLY_FILES     = []                                     # e.g. ["WELD"] to process only matching file names
AUTO_TEMPLATE  = True                                   # when the FILE NAME matches no template, work the form out
                                                        # from the page itself (Stage 7b): the number of columns in
                                                        # the printed grid, and the heading printed above the table
OUTPUT_DIR     = f"{DRIVE_ROOT}/ocr_output_v3" if IN_COLAB else "ocr_output"   # crops / debug / JSON / CSV
COPY_OUTPUT_TO_DRIVE = None                             # not needed on Colab any more: OUTPUT_DIR is already on Drive

# ---- rendering / pre-processing ----------------------------------------------
GOT_MODEL_ID   = "stepfun-ai/GOT-OCR-2.0-hf"   # the OCR model (Stage 7b may need it before Stage 14 does)

+    = "pymupdf"      # "pymupdf" | "pdf2image" (needs poppler; installed automatically on Colab)
DPI            = 300            # requirement: 300 DPI page images
DENOISE_H      = 7              # fastNlMeans strength (higher = smoother, may blur thin pen strokes)
CLAHE_CLIP     = 2.0            # contrast enhancement strength
MAX_DESKEW_DEG = 5.0            # ignore |skew| larger than this (would mean a bad estimate)

# ---- table / grid detection ---------------------------------------------------
LINE_KERNEL_IN = 0.40           # min length (inches) of a printed line for the morphological line masks
ROW_COVERAGE   = 0.50           # a y-position is a row line if it is inked over >= 50% of the table width
COL_COVERAGE   = 0.50           # (starting value; the detector sweeps it down until the column count matches)

# ---- crop geometry  (IMPORTANT R&D REQUIREMENT) --------------------------------
PADDING        = 10             # px added left/right of every column crop so strokes crossing the grid are not cut
                                # (this is the MINIMUM; see CROP_MAX_OVERFLOW below)
CROP_MAX_OVERFLOW = 40          # px a crop may grow beyond the printed cell to catch handwriting that overflows the
                                # column. The QTY columns are only ~67 px wide at 300 DPI, so a 4-digit number
                                # written in one routinely spills over the printed line and used to be cut in half.
CROP_GUTTER_PX    = 6           # px of white that ends the overflow — the gap that separates this entry from the
                                # neighbouring column's handwriting
CROP_INK_EPS      = 1           # ink pixels in a column below this counts as white
CROP_EDGE_MARGIN  = 3           # px kept beyond the last inked column
V_PADDING      = 10             # px added above/below (handwriting also crosses horizontal lines; lower it if
                                # the neighbouring record bleeds into tight rows)
V_PADDING_TOP   = 2             # px above the FIRST data row when header text sits right above it (the minimum)
V_PADDING_FIRST = 14            # px the FIRST data row may grow upwards when there is room. Handwriting on the
                                # first line often rises above the header/body boundary and used to be chopped by
                                # it; the crop now stops just below any header text found in the gap, so the
                                # printed column names are still not pulled in.
LAST_COL_MODE  = "page"         # last column right edge:
                                #   "page"  -> extend all the way to the right edge of the page image (default)
                                #   "table" -> table_right + PADDING (the strict formula)

# ---- line removal / ink tests -------------------------------------------------
LINE_REMOVAL_MODE = "guided"    # "guided": morphological lines, but only near the known grid positions
                                # "morph" : morphological lines anywhere in the crop (also erases handwritten underlines)
LINE_KERNEL_FRAC  = 0.55        # a run must span this fraction of the crop to count as a printed grid line
LINE_RUN_CAP_PX   = 60          # ... but never has to be longer than this, so the short line stub in the very wide
                                # last column is still recognised as a line
LINE_GUIDE_TOL    = 8           # px: how far from its expected position a printed line is searched for in a crop
LINE_HALF_PX      = 5           # px: half-thickness of the band around each printed line that MAY be erased
LINE_CROSS_PX     = 8           # px searched above/below (and left/right of) a printed line for the CONTINUATION
                                # of a pen stroke. Only a real crossing - ink on BOTH sides - is protected from
                                # the erase. Text merely resting ON a line has ink on one side only, so the line
                                # under it is still removed. Raise if letters break, lower if lines survive.
LINE_INPAINT_RAD  = 2           # px: cv2.inpaint radius used to fill the erased line pixels
INK_GRAY_THRESH   = 150         # gray level below which a pixel counts as pen ink (after flatten + CLAHE)
MIN_INK_PIXELS    = 120         # fewer ink pixels than this in a cell -> treated as empty
MIN_ANCHOR_INK    = 150         # ink needed in the anchor column to start a new record

# ---- debugging ----------------------------------------------------------------
SHOW_DEBUG     = True           # display intermediate images inline
SAVE_DEBUG     = True           # also save them to OUTPUT_DIR/debug/<page>/
VIEW_PAGES     = None           # None = show all pages; or e.g. ["SUPPLIER REJECTION REPORT_p1"]

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ---- utilities used by several stages ------------------------------------------
def safe_name(s):
    return re.sub(r"[^A-Za-z0-9]+", "_", str(s)).strip("_")

def to_bgr(img):
    return cv2.cvtColor(img, cv2.COLOR_GRAY2BGR) if img.ndim == 2 else img

def show(img, title="", figsize=(9, 12), max_side=None):
    """Display a BGR / gray image inline (nothing happens if SHOW_DEBUG is False)."""
    if not SHOW_DEBUG:
        return
    plt.figure(figsize=figsize)
    if img.ndim == 2:
        plt.imshow(img, cmap="gray", vmin=0, vmax=255)
    else:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title, fontsize=11)
    plt.axis("off")
    plt.tight_layout()
    plt.show()

def save_debug(page, name, img):
    """Write a debug image to OUTPUT_DIR/debug/<page uid>/<name>."""
    if not SAVE_DEBUG:
        return None
    d = os.path.join(OUTPUT_DIR, "debug", page.uid)
    os.makedirs(os.path.dirname(os.path.join(d, name)), exist_ok=True)
    path = os.path.join(d, name)
    cv2.imwrite(path, img)
    return path

def group_peaks(profile, thr, min_gap=6):
    """Centres (intensity-weighted) of runs where profile >= thr; runs closer than min_gap px merge."""
    idx = np.flatnonzero(profile >= thr)
    if idx.size == 0:
        return []
    groups = np.split(idx, np.flatnonzero(np.diff(idx) > min_gap) + 1)
    return [float((profile[g] * g).sum() / profile[g].sum()) for g in groups]

def viewed(page):
    return VIEW_PAGES is None or page.uid in VIEW_PAGES

print("Config loaded. Input mode:", INPUT_MODE, "| DPI:", DPI, "| padding:", PADDING, "| last column:", LAST_COL_MODE)
print("Uploads ->", UPLOAD_DIR)
print("Output  ->", OUTPUT_DIR, "(permanent Drive storage)" if IN_COLAB else "(local folder)")


ValueError: mount failed

In [ ]:
# ============================================================================
# STAGE 2 — FORM TEMPLATES  (configurable column / header names per form)
#
#   match               : upper-case fragments looked for in the PDF file name (first template that matches wins)
#   columns             : header names, LEFT -> RIGHT.  The OCR result is stored under these names.
#   anchor              : column that is filled on the FIRST printed row of every record
#                         (DATE / S.NO).  A new record starts wherever the anchor cell has ink.  None = one record per inked row.
#   max_rows_per_record : how many printed grid rows one handwritten record may span (wrapped text)
#
# To support a new form, add an entry here.  To force a template for a file, use TEMPLATE_BY_FILE.
# ============================================================================
TEMPLATES = {
    "supplier_lot_rejection_summary": dict(
        match=["LOT REJECTION"],
        columns=["DATE", "PART NAME", "SUPPLIER NAME", "LOT QTY", "REW QTY", "REJ QTY", "OK QTY",
                 "PROBLEM DESCRIPTION", "ACTION", "RETURN STATUS", "CAPA STATUS", "REMARKS"],
        anchor="DATE", max_rows_per_record=2),
    "weld_shop_rejection_report": dict(
        match=["WELD SHOP", "WELDSHOP"],
        columns=["S.NO", "ITEM CODE", "MATERIAL PARTICULARS", "UNIT", "REJECTION QTY", "VENDOR NAME", "REASON / REMARKS"],
        anchor="S.NO", max_rows_per_record=2),
    "supplier_rework_report": dict(
        match=["REWORK"],
        columns=["DATE", "PART NAME", "SUPPLIER NAME", "LOT QTY", "REWORK QTY", "OK QTY", "PROBLEM DESCRIPTION", "REMARKS"],
        anchor="DATE", max_rows_per_record=2),
    "supplier_segregation_report": dict(
        match=["SEGREGATION"],
        columns=["DATE", "PART NAME", "SUPPLIER NAME", "LOT QTY", "SEGREGATION QTY", "OK QTY", "PROBLEM DESCRIPTION", "REMARKS"],
        anchor="DATE", max_rows_per_record=2),
    "supplier_rejection_report": dict(
        match=["SUPPLIER REJECTION"],
        columns=["DATE", "PART NAME", "SUPPLIER NAME", "LOT QTY", "REJECTION QTY", "OK QTY", "PROBLEM DESCRIPTION", "REMARKS"],
        anchor="DATE", max_rows_per_record=2),
}

# Force a template for a specific file (file name -> template key), e.g. scans with generic names:
TEMPLATE_BY_FILE = {
    # "scan_0042.pdf": "supplier_rework_report",
}

def pick_template(pdf_path):
    name = os.path.basename(pdf_path)
    if name in TEMPLATE_BY_FILE:
        return TEMPLATE_BY_FILE[name]
    norm = re.sub(r"[_\-\.]+", " ", os.path.splitext(name)[0]).upper()
    for key, t in TEMPLATES.items():
        if any(m in norm for m in t["match"]):
            return key
    return None

print("Templates:", {k: len(v["columns"]) for k, v in TEMPLATES.items()})


In [ ]:
# ============================================================================
# STAGE 3 — MANUAL CONFIGURATION  (use this when automatic grid detection fails)
#
# Keyed by PDF *stem* (file name without .pdf, checked first) or by template key.
# Coordinates are pixels of the ALIGNED 300-DPI page — the same numbers printed in the Stage 9 log and
# drawn on the "row/column boundaries" debug image, so you can read them off and correct them.
#
#   col_x        : N+1 x-positions (every vertical line, left border ... right border)
#                  or N x-positions (left edges only; the table's right edge is then used as the last boundary)
#   header_names : optional, overrides the template's column names for that page
#   body_top     : y of the line that separates the header row from the first data row
#   body_bottom  : y of the last data-row line (top of the signature/footer row)
#   row_y        : optional explicit list of every horizontal line in the data area
#                  (if you only give body_top/body_bottom + row_pitch, lines are generated at that pitch)
#   row_pitch    : optional uniform row height in px
#   anchor / max_rows_per_record : optional overrides of the template's record-grouping settings
#
# Example (values are illustrative — leave MANUAL_CONFIG empty unless auto-detection misbehaves):
#   "SUPPLIER LOT REJECTION SUMMARY": dict(
#       col_x=[205, 309, 529, 706, 773, 841, 909, 975, 1655, 1889, 2023, 2111, 2229],
#       body_top=354, body_bottom=2004, row_pitch=64),
# ============================================================================
MANUAL_CONFIG = {
}


In [ ]:
# ============================================================================
# STAGE 4 — LOAD INPUT PDFs   (Google Drive folder | file upload | local folder)
# ============================================================================
def load_pdf_paths():
    if INPUT_MODE == "drive":
        mount_drive()                                        # already mounted in Stage 1; no-op here
        folder = DRIVE_PDF_DIR
    elif INPUT_MODE == "upload":
        from google.colab import files
        mount_drive()                                        # UPLOAD_DIR is on Drive, so the PDFs are kept
        os.makedirs(UPLOAD_DIR, exist_ok=True)
        print("Select the PDF files to upload ...")
        for fname, data in files.upload().items():           # opens the browser file picker
            with open(os.path.join(UPLOAD_DIR, fname), "wb") as f:
                f.write(data)
        folder = UPLOAD_DIR
    else:
        folder = LOCAL_PDF_DIR
    paths = sorted({os.path.normcase(p): p for p in glob.glob(os.path.join(folder, "*")) if p.lower().endswith(".pdf")}.values())
    if ONLY_FILES:
        paths = [p for p in paths if any(s.upper() in os.path.basename(p).upper() for s in ONLY_FILES)]
    return paths

PDF_PATHS = load_pdf_paths()
print(f"{len(PDF_PATHS)} PDF(s) found:")
for p in PDF_PATHS:
    key = pick_template(p)
    print(f"  {os.path.basename(p):45s} -> template: {key or 'NO MATCH (will be skipped)'}")


In [ ]:
# ============================================================================
# STAGE 5 — PDF  ->  300 DPI IMAGES
# PyMuPDF rasterises each page at exactly DPI dots-per-inch (page size in points / 72 * DPI).
# pdf2image (poppler) is kept as a fallback backend.
# ============================================================================
try:
    import pymupdf
except ImportError:                       # older PyMuPDF releases expose the module as "fitz"
    import fitz as pymupdf


@dataclass
class Page:
    """Everything we learn about one PDF page, filled in stage by stage."""
    pdf: str                              # source PDF file name
    page_no: int                          # 1-based page number
    uid: str                              # unique id used for debug folders, e.g. "WELD SHOP REJECTION REPORT_p1"
    template_key: str
    template: dict
    original: np.ndarray                  # BGR page image at DPI (untouched)
    angle: float = 0.0                    # deskew rotation applied (degrees)
    aligned: Optional[np.ndarray] = None  # BGR, deskewed
    flat: Optional[np.ndarray] = None     # gray, denoised + illumination-flattened + deskewed (no CLAHE)
    gray: Optional[np.ndarray] = None     # gray, + CLAHE contrast — this is what the crops are cut from
    bw: Optional[np.ndarray] = None       # binary (ink = 255) used for grid detection
    hmask: Optional[np.ndarray] = None    # horizontal-line mask
    vmask: Optional[np.ndarray] = None    # vertical-line mask
    table_bbox: Optional[tuple] = None    # (x1, y1, x2, y2)
    detect: dict = field(default_factory=dict)   # raw detection results (rows, cols, thresholds ...)
    grid: dict = field(default_factory=dict)     # resolved grid: col_x, col_names, row_y ...
    records: list = field(default_factory=list)  # grouped records
    warnings: list = field(default_factory=list)
    result: dict = field(default_factory=dict)


def render_pdf(path, dpi=DPI):
    """Return a list of BGR page images."""
    if PDF_BACKEND == "pdf2image":
        if IN_COLAB:
            subprocess.run(["apt-get", "-qq", "install", "-y", "poppler-utils"], check=False)
        ensure("pdf2image")
        from pdf2image import convert_from_path
        return [cv2.cvtColor(np.array(im.convert("RGB")), cv2.COLOR_RGB2BGR) for im in convert_from_path(path, dpi=dpi)]
    pages = []
    with pymupdf.open(path) as doc:
        for pg in doc:
            pix = pg.get_pixmap(dpi=dpi, colorspace=pymupdf.csRGB, alpha=False)
            arr = np.frombuffer(pix.samples, np.uint8).reshape(pix.height, pix.width, 3)
            pages.append(cv2.cvtColor(arr, cv2.COLOR_RGB2BGR))
    return pages


PAGES = []
for path in PDF_PATHS:
    key = pick_template(path)
    if key is None and not AUTO_TEMPLATE:
        print(f"skip {os.path.basename(path)}: no template matches (add one in Stage 2)")
        continue
    stem = os.path.splitext(os.path.basename(path))[0]
    for i, img in enumerate(render_pdf(path), start=1):
        PAGES.append(Page(pdf=os.path.basename(path), page_no=i, uid=f"{stem}_p{i}", template_key=key,
                          template=TEMPLATES[key] if key else {}, original=img))
        how = f"template: {key}" if key else "template: to be identified from the page (Stage 7b)"
        print(f"rendered {stem} p{i}: {img.shape[1]}x{img.shape[0]} px @ {DPI} DPI  |  {how}")

for pg in PAGES:
    if viewed(pg):
        show(pg.original, f"ORIGINAL PAGE — {pg.uid}", figsize=(7, 10))
        save_debug(pg, "01_original.jpg", pg.original)


In [ ]:
# ============================================================================
# STAGE 6 — PRE-PROCESSING
#   1. grayscale
#   2. denoise            (fast non-local means — removes scanner speckle but keeps pen strokes)
#   3. flatten lighting   (divide by an estimate of the paper background -> even white page, no shadows/bands)
#   4. deskew / de-shear  (rotation from the long horizontal lines + shear from the vertical lines, applied as ONE
#                          affine warp so scans of photocopied forms end up with truly level rows and upright columns)
#   5. contrast (CLAHE)   (local contrast enhancement so faint pen strokes become dark)
# Grid detection uses `flat`; the crops that go to OCR are cut from `gray` (= flat + CLAHE).
# ============================================================================
def flatten_illumination(gray):
    """Divide by the paper background. The background is estimated on a 1/4-size copy (fast) with a
    morphological closing that erases thin dark things (pen strokes, printed lines) and keeps the paper."""
    small = cv2.resize(gray, None, fx=0.25, fy=0.25, interpolation=cv2.INTER_AREA)
    bg = cv2.morphologyEx(small, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15)))
    bg = cv2.GaussianBlur(bg, (0, 0), 6)
    bg = cv2.resize(bg, (gray.shape[1], gray.shape[0]), interpolation=cv2.INTER_LINEAR)
    return cv2.divide(gray, np.maximum(bg, 1), scale=255)


def line_masks(gray, dpi=DPI):
    """Binary ink image + masks of long horizontal / vertical strokes (the printed grid)."""
    bw = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 35, 15)
    k = int(LINE_KERNEL_IN * dpi)                       # a printed line is much longer than any pen stroke
    h = cv2.morphologyEx(bw, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (k, 1)))
    v = cv2.morphologyEx(bw, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (1, k)))
    return bw, h, v


def _rot_matrix(angle, w, h):
    """3x3 rotation about the page centre; positive angle = counter-clockwise."""
    return np.vstack([cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0), [0, 0, 1]])


def _shear_matrix(tilt, w, h):
    """3x3 horizontal shear about the middle row: x' = x - tan(tilt) * (y - h/2). Straightens verticals that lean right by `tilt`."""
    t = math.tan(math.radians(tilt))
    return np.array([[1, -t, t * h / 2], [0, 1, 0], [0, 0, 1.0]])


def _sharpness(mask, M, axis):
    """Sum of squared projection: largest when all line pixels stack into a few sharp peaks (= perfectly straight lines)."""
    h, w = mask.shape
    p = cv2.warpAffine(mask, M[:2], (w, h), flags=cv2.INTER_NEAREST).sum(axis=axis).astype(np.float64)
    return float((p * p).sum())


def _maximise(fn, lo, hi):
    """Coarse-to-fine 1-D search for the angle that maximises fn."""
    best = 0.0
    for step in (0.2, 0.05, 0.01):
        best = max(np.arange(lo, hi + 1e-9, step), key=fn)
        lo, hi = best - step, best + step
    return float(best)


def align_matrix(flat0):
    """Affine transform that levels the printed rows (rotation) and makes the printed columns upright (shear).
    Both angles are found by projection-profile search on a half-size line mask: the right angle is the one where
    the row (or column) profile is sharpest. Returns (3x3 matrix, rotation deg, shear deg)."""
    small = cv2.resize(flat0, None, fx=0.5, fy=0.5, interpolation=cv2.INTER_AREA)
    h, w = small.shape
    bw = cv2.adaptiveThreshold(small, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 25, 8)
    hm = cv2.morphologyEx(bw, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (40, 1)))
    vm = cv2.morphologyEx(bw, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (1, 20)))
    rot = _maximise(lambda a: _sharpness(hm, _rot_matrix(a, w, h), axis=1), -MAX_DESKEW_DEG, MAX_DESKEW_DEG)
    R = _rot_matrix(rot, w, h)
    shear = _maximise(lambda t: _sharpness(vm, _shear_matrix(t, w, h) @ R, axis=0), -1.0, 1.0)
    H, W = flat0.shape
    return _shear_matrix(shear, W, H) @ _rot_matrix(rot, W, H), rot, shear


def warp(img, M, border):
    h, w = img.shape[:2]
    return cv2.warpAffine(img, M[:2], (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_CONSTANT, borderValue=border)


def preprocess(page):
    gray0 = cv2.cvtColor(page.original, cv2.COLOR_BGR2GRAY)                           # 1. grayscale
    den = cv2.fastNlMeansDenoising(gray0, None, h=DENOISE_H, templateWindowSize=5, searchWindowSize=11)   # 2. denoise
    flat0 = flatten_illumination(den)                                                 # 3. even lighting
    M, rot, shear = align_matrix(flat0)                                               # 4. deskew + de-shear (one warp)
    flat = warp(flat0, M, 255)
    page.aligned = warp(page.original, M, (255, 255, 255))
    page.angle = rot
    page.flat = flat
    page.gray = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=(8, 8)).apply(flat)   # 5. contrast
    page.bw, page.hmask, page.vmask = line_masks(flat)
    print(f"{page.uid}: rotated {rot:+.2f} deg, sheared {shear:+.2f} deg")


for pg in PAGES:
    t0 = time.time()
    preprocess(pg)
    if viewed(pg):
        show(np.hstack([cv2.resize(cv2.cvtColor(cv2.cvtColor(pg.original, cv2.COLOR_BGR2GRAY), cv2.COLOR_GRAY2BGR), None, fx=0.25, fy=0.25),
                        cv2.resize(to_bgr(pg.gray), None, fx=0.25, fy=0.25)]),
             f"PRE-PROCESSED — {pg.uid}   (left: original gray | right: denoised + deskewed + contrast)", figsize=(12, 8))
        save_debug(pg, "02_preprocessed.png", pg.gray)


In [ ]:
# ============================================================================
# STAGE 7 — TABLE DETECTION / ALIGNMENT
# The printed grid lines (horizontal | vertical masks from Stage 6) are merged; the biggest connected blob is the
# table. Its bounding box gives the table's left/top/right/bottom. (The page is already rotation-aligned by Stage 6.)
# ============================================================================
def detect_table(page):
    grid_mask = cv2.dilate(cv2.bitwise_or(page.hmask, page.vmask), np.ones((9, 9), np.uint8))
    n, _, stats, _ = cv2.connectedComponentsWithStats(grid_mask)
    if n < 2:
        raise RuntimeError("no table-like structure found")
    k = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    x, y, w, h = stats[k, :4]
    return (int(x), int(y), int(x + w), int(y + h))


def draw_table(page):
    vis = page.aligned.copy()
    x1, y1, x2, y2 = page.table_bbox
    cv2.rectangle(vis, (x1, y1), (x2, y2), (255, 0, 0), 6)
    for (cx, cy) in [(x1, y1), (x2, y1), (x1, y2), (x2, y2)]:
        cv2.circle(vis, (cx, cy), 18, (0, 0, 255), -1)
    cv2.putText(vis, f"table {x2-x1}x{y2-y1}px", (x1 + 10, max(40, y1 - 15)), cv2.FONT_HERSHEY_SIMPLEX, 1.6, (255, 0, 0), 4)
    return vis


for pg in PAGES:
    try:
        pg.table_bbox = detect_table(pg)
    except Exception as e:
        pg.warnings.append(f"table detection failed: {e}")
        print(f"{pg.uid}: TABLE NOT FOUND — {e}")
        continue
    print(f"{pg.uid}: table bbox = {pg.table_bbox}")
    vis = draw_table(pg)
    save_debug(pg, "03_table_detected.png", vis)
    save_debug(pg, "03b_line_masks.png", cv2.bitwise_or(pg.hmask, pg.vmask))
    if viewed(pg) and SHOW_DEBUG:
        fig, ax = plt.subplots(1, 2, figsize=(14, 9))
        ax[0].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); ax[0].set_title(f"DETECTED TABLE — {pg.uid}")
        ax[1].imshow(cv2.bitwise_or(pg.hmask, pg.vmask), cmap="gray"); ax[1].set_title("line masks (horizontal | vertical)")
        for a in ax: a.axis("off")
        plt.tight_layout(); plt.show()


In [ ]:
# ============================================================================
# STAGE 7b — IDENTIFY THE FORM FROM THE PAGE  (only for files the name did not identify)
# The file name is a poor key: a scan called "HandwrittenReport.pdf" matches no template and used to be skipped.
# Two signals on the page itself are enough, and they are tried in that order because the first is free:
#   1. STRUCTURE - how many columns the printed grid has. On these forms that alone settles it whenever the count
#      is unique (the lot-rejection form has 12, the weld-shop one 7).
#   2. HEADING   - when several templates share a column count (rework / segregation / supplier-rejection all have
#      8), the block printed above the first data row is read with the OCR model and matched against each
#      template's `match` fragments and column names. That text is PRINTED, so it reads far more reliably than the
#      handwriting does.
# A page that still cannot be identified keeps template = {} and is skipped by Stage 8 with a warning, rather than
# being forced into the wrong template.
# ============================================================================
def ensure_got():
    """Load GOT-OCR-2.0 once, into module globals. Safe to call again - Stage 14 calls it too."""
    global got_processor, got_model, device
    if "got_model" in globals() and got_model is not None:
        return
    import torch as _torch
    from transformers import AutoProcessor, AutoModelForImageTextToText
    globals()["torch"] = _torch
    device = "cuda" if _torch.cuda.is_available() else "cpu"
    if device == "cpu":
        print("WARNING: no GPU — GOT-OCR-2.0 will be slow on CPU.")
    print("Loading GOT-OCR-2.0 ...")
    got_processor = AutoProcessor.from_pretrained(GOT_MODEL_ID)
    got_model = AutoModelForImageTextToText.from_pretrained(
        GOT_MODEL_ID, device_map="auto", dtype=_torch.float16 if device == "cuda" else _torch.float32)
    got_model.eval()


def got_generate(img, max_new_tokens, no_repeat_ngram=0):
    """Run GOT on one grayscale image -> (text, mean per-token probability). The one place generate() is called."""
    ensure_got()
    image = Image.fromarray(img).convert("RGB")
    inputs = got_processor(image, return_tensors="pt")
    inputs = {k: v.to(got_model.device) if hasattr(v, "to") else v for k, v in inputs.items()}
    with torch.inference_mode():
        out = got_model.generate(**inputs, do_sample=False, tokenizer=got_processor.tokenizer,
                                 stop_strings="<|im_end|>", max_new_tokens=max_new_tokens,
                                 no_repeat_ngram_size=no_repeat_ngram or 0,
                                 output_scores=True, return_dict_in_generate=True)
    new_ids = out.sequences[0][inputs["input_ids"].shape[1]:]
    text = got_processor.decode(new_ids, skip_special_tokens=True).strip()
    special = set(got_processor.tokenizer.all_special_ids)
    probs = [torch.softmax(step[0], dim=-1)[t].item() for step, t in zip(out.scores, new_ids)
             if t.item() not in special]
    return text, (float(np.mean(probs)) if probs else None)


def count_columns(page):
    """How many columns the printed grid has, WITHOUT knowing the form. None if no stable answer.

    Stage 8 sweeps the coverage threshold until the line count matches the template. Here there is nothing to match,
    so the count that survives the widest range of thresholds wins: a real grid holds its line count over many
    thresholds, while a count produced by noise appears at one threshold and vanishes at the next.
    """
    x1, y1, x2, y2 = page.table_bbox
    H = y2 - y1
    band = page.vmask[y1 + int(0.25 * H): y1 + int(0.85 * H), max(x1 - 5, 0): x2 + 6] > 0
    if band.size == 0:
        return None
    cover = band.sum(0) / band.shape[0]
    votes = {}
    for thr in np.arange(0.80, 0.14, -0.02):
        n = len(group_peaks(cover, float(thr)))
        if n >= 4:                                          # a form has at least 3 columns
            votes[n] = votes.get(n, 0) + 1
    if not votes:
        return None
    lines = max(votes.items(), key=lambda kv: kv[1])[0]
    return lines - 1                                        # N+1 printed lines -> N columns


def read_form_heading(page):
    """The printed block above the data rows (company name, form title, column headers) as one string."""
    x1, y1, x2, y2 = page.table_bbox
    top = max(y1 - int(0.05 * page.gray.shape[0]), 0)       # some forms print the title just outside the grid
    bottom = min(y1 + int(0.22 * (y2 - y1)), y2)
    band = page.gray[top:bottom, x1:x2]
    if band.size == 0:
        return ""
    scale = min(1.0, 1000.0 / max(band.shape[1], 1))        # a full-width banner is wider than the model needs
    if scale < 1.0:
        band = cv2.resize(band, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    text, _ = got_generate(band, 256)
    return text


def _norm_words(s):
    return re.sub(r"\s+", " ", re.sub(r"[^A-Z0-9 ]+", " ", str(s).upper())).strip()


def template_score(heading, template):
    """How well a heading matches one template: its `match` fragments count most, its column names confirm."""
    text = _norm_words(heading)
    score = 5 * sum(1 for m in template["match"] if _norm_words(m) in text)
    score += sum(1 for c in template["columns"] if len(_norm_words(c)) > 2 and _norm_words(c) in text)
    return score


def identify_template(page):
    """(template key, why) for a page whose file name told us nothing. (None, why) when it stays unknown."""
    n = count_columns(page)
    candidates = [k for k, t in TEMPLATES.items() if len(t["columns"]) == n] if n else []
    if len(candidates) == 1:
        return candidates[0], f"{n} columns - only {candidates[0]} has that many"

    heading = read_form_heading(page)
    pool = candidates or list(TEMPLATES)
    ranked = sorted(((template_score(heading, TEMPLATES[k]), k) for k in pool), reverse=True)
    best, key = ranked[0]
    if best <= 0:
        return None, f"{n} columns, heading {heading[:50]!r} matched no template"
    runner = ranked[1][0] if len(ranked) > 1 else 0
    return key, (f"{n} columns -> {len(pool)} candidates; heading matched {key} "
                 f"(score {best} vs next {runner})")


for pg in PAGES:
    if pg.template or pg.table_bbox is None:
        continue
    key, why = identify_template(pg)
    if key:
        pg.template_key, pg.template = key, TEMPLATES[key]
        print(f"{pg.uid}: identified as {key}  ({why})")
    else:
        pg.warnings.append(f"form not identified: {why}")
        print(f"{pg.uid}: COULD NOT IDENTIFY THE FORM ({why})\n"
              f"    -> add a template in Stage 2, or pin it with TEMPLATE_BY_FILE[{pg.pdf!r}]")


In [ ]:
# ============================================================================
# STAGE 8 — DETECT HORIZONTAL ROWS AND VERTICAL COLUMNS
# Projection profiles: for every x we measure what fraction of the table body is covered by the vertical-line mask
# (and for every y the fraction of the table width covered by the horizontal-line mask). Real grid lines are
# ~55-65 % covered, handwriting/noise is < 25 %.
#   * columns : the coverage threshold is swept downward until we get exactly N+1 vertical lines (N = names in template)
#   * rows    : darkness profile of the table width (robust to faint / curved scanned lines); the header/body split is the
#               bottom of the first row band that every column line crosses, the data area ends where the column lines end
#   * missing faint row lines are re-inserted when a gap is (almost) an integer multiple of the row pitch
# ============================================================================
def get_manual(page):
    """Manual override dict for this page (PDF stem wins over template key)."""
    stem = os.path.splitext(page.pdf)[0]
    return MANUAL_CONFIG.get(stem) or MANUAL_CONFIG.get(page.template_key) or {}


def detect_columns(page, n_expected):
    x1, y1, x2, y2 = page.table_bbox
    H = y2 - y1
    band = page.vmask[y1 + int(0.25 * H): y1 + int(0.85 * H), max(x1 - 5, 0): x2 + 6] > 0   # middle of the body
    cover = band.sum(0) / band.shape[0]
    off = max(x1 - 5, 0)
    best = None
    for thr in np.arange(0.80, 0.14, -0.05):
        pk = [p + off for p in group_peaks(cover, thr)]
        if len(pk) == n_expected + 1:
            return pk, float(thr)
        if best is None or abs(len(pk) - (n_expected + 1)) < abs(len(best[0]) - (n_expected + 1)):
            best = (pk, float(thr))
    return best


def line_vertical_extent(page, xs):
    """Top / bottom y of each vertical line (uses the vertical mask in a +-5 px window around x)."""
    x1, y1, x2, y2 = page.table_bbox
    tops, bottoms = [], []
    for x in xs:
        xi = int(round(x))
        ys = np.flatnonzero((page.vmask[y1: y2 + 1, max(xi - 5, 0): xi + 6] > 0).any(axis=1))
        tops.append(y1 + int(ys[0]) if ys.size else y1)
        bottoms.append(y1 + int(ys[-1]) if ys.size else y2)
    return tops, bottoms


def row_line_positions(page, x1, x2, y1, y2, n_chunks=16, pct=35, min_cover=0.20):
    """y-positions of the printed horizontal lines between y1 and y2. Two independent signals must agree:
      (a) morphological : fraction of the table width covered by the long-horizontal-run mask (>= min_cover). Text rows
                          are ~0, good lines ~0.6, faint scanned lines still ~0.3.
      (b) darkness      : (255 - flattened gray) of the table width, split into n_chunks slabs; the pct-th percentile
                          ACROSS slabs is high only where a stroke crosses most of the width (a printed line), so
                          handwriting confined to a few cells and header text do not qualify.
    Requiring both is robust to faint / slightly curved lines (a) and to bold header text (b)."""
    y1, y2 = max(y1, 0), min(y2, page.flat.shape[0])
    band = page.hmask[y1:y2, x1:x2] > 0
    cover = band.sum(1) / band.shape[1]
    cover = np.maximum.reduce([np.roll(cover, s) for s in (-2, -1, 0, 1, 2)])          # tolerate 1-2 px of line drift
    reg = 255.0 - page.flat[y1:y2, x1:x2].astype(np.float32)
    dark = np.percentile(np.stack([c.mean(axis=1) for c in np.array_split(reg, n_chunks, axis=1)]), pct, axis=0)
    dark = np.convolve(dark, np.ones(3) / 3, mode="same")
    dark = np.maximum.reduce([np.roll(dark, s) for s in (-2, -1, 0, 1, 2)])
    lines = [p for p in group_peaks(cover, min_cover, min_gap=6) if dark[int(round(p))] >= 10.0]
    return [p + y1 for p in lines], min_cover


def band_has_all_cols(page, ya, yb, cols, min_frac=0.85):
    """True if (almost) every column line crosses the horizontal band ya..yb — used to find the header-name row.
    A +-8 px window and min_frac < 1 tolerate a line that is a few px off or lost in one header cell."""
    ya, yb = int(ya) + 6, int(yb) - 6
    if yb <= ya:
        return False
    ok = 0
    for x in cols:
        xi = int(round(x))
        ok += (page.vmask[ya:yb, max(xi - 8, 0): xi + 9] > 0).any(axis=1).mean() >= 0.7
    return ok >= min_frac * len(cols)


def fill_missing_rows(lines):
    """Insert lines where a gap is ~k x the median pitch (k = 2..4) — a faint / broken printed line."""
    if len(lines) < 4:
        return lines
    pitch = float(np.median(np.diff(lines)))
    out = [lines[0]]
    for a, b in zip(lines[:-1], lines[1:]):
        k = int(round((b - a) / pitch))
        if k >= 2 and abs((b - a) - k * pitch) < 0.18 * pitch:
            out += [a + (b - a) * j / k for j in range(1, k)]
        out.append(b)
    return out


def detect_rows_and_cols(page):
    man = get_manual(page)
    names = man.get("header_names") or page.template["columns"]
    n = len(names)
    cols, thr = detect_columns(page, n)
    x1, y1, x2, y2 = page.table_bbox
    d = dict(cols=cols, col_thr=thr, n_expected=n)

    # where the column lines end -> bottom of the data area (top of the signature / footer row)
    interior = cols[1:-1] if len(cols) > 2 else cols
    _, bottoms = line_vertical_extent(page, interior)
    body_end = float(np.median(bottoms))

    # all horizontal grid lines across the width spanned by the columns
    rows_all, rthr = row_line_positions(page, int(min(cols)) + 8, int(max(cols)) - 8, y1 - 6, y2 + 7)

    # header-name row = first band crossed by ALL column lines; the data area starts at its bottom line
    body_top = None
    for a, b_ in zip(rows_all[:-1], rows_all[1:]):
        if b_ - a > 25 and band_has_all_cols(page, a, b_, cols):
            body_top = b_
            d["names_band"] = (a, b_)
            break
    body = [r for r in rows_all if body_top is not None and body_top - 6 <= r <= body_end + 15]
    body = fill_missing_rows(body)
    if len(body) >= 4:                                     # faint tail: continue at the median pitch down to the footer line
        pitch = float(np.median(np.diff(body)))
        while body_end - body[-1] > 0.6 * pitch:
            body.append(body[-1] + pitch)
        if abs(body[-1] - body_end) < 0.4 * pitch:
            body[-1] = body_end
    d.update(body_end=body_end, rows_all=rows_all, row_thr=rthr, body_top=body_top, body_rows=body)
    page.detect = d
    return d


for pg in PAGES:
    if pg.table_bbox is None or not pg.template:
        continue
    d = detect_rows_and_cols(pg)
    print(f"{pg.uid}: {len(d['cols'])} vertical lines (expected {d['n_expected'] + 1}, coverage thr {d['col_thr']:.2f}) | "
          f"header-name row {d.get('names_band')}, body_top={d['body_top']}, {max(len(d['body_rows']) - 1, 0)} body rows")


In [ ]:
# ============================================================================
# STAGE 9 — RESOLVE THE GRID  (auto-detection + template names + manual overrides)
# Produces page.grid = {col_x, col_names, row_y, table_right ...}. Anything in MANUAL_CONFIG (Stage 3) wins.
# If the number of vertical lines does not match the template, the page is flagged and skipped by later stages —
# fill MANUAL_CONFIG for it and re-run from this stage.
# ============================================================================
def resolve_grid(page):
    man = get_manual(page)
    d = page.detect
    names = list(man.get("header_names") or page.template["columns"])
    n = len(names)
    x1, y1, x2, y2 = page.table_bbox
    notes, src = [], {}

    # ---- columns -----------------------------------------------------------------------------------------
    if man.get("col_x"):
        col_x = [float(v) for v in man["col_x"]]
        if len(col_x) == n:
            col_x.append(float(x2))                          # left edges only -> table right edge closes the last column
        src["cols"] = "manual"
    else:
        col_x = list(d["cols"])
        if len(col_x) == n:                                  # right border line not detected -> use table bbox edge
            col_x.append(float(x2)); notes.append("right border not detected; using table bbox edge")
        src["cols"] = "auto"
    if len(col_x) != n + 1:
        raise ValueError(f"found {len(col_x)} vertical lines for {n} columns ({n + 1} needed). "
                         f"Detected x = {[int(v) for v in col_x]}. Set MANUAL_CONFIG col_x for '{os.path.splitext(page.pdf)[0]}'.")
    col_x = sorted(col_x)

    # ---- rows ---------------------------------------------------------------------------------------------
    body_top = man.get("body_top", d["body_top"])
    if body_top is None:
        raise ValueError("could not find the header/body boundary; set body_top in MANUAL_CONFIG")
    if man.get("row_y"):
        row_y = sorted(float(v) for v in man["row_y"]); src["rows"] = "manual"
    else:
        rows = list(d["body_rows"])
        if "body_bottom" in man or "body_top" in man:
            lo, hi = body_top - 6, man.get("body_bottom", rows[-1] if rows else body_top) + 6
            rows = [r for r in d["rows_all"] if lo <= r <= hi]
        if man.get("row_pitch"):                             # uniform grid between body_top and body_bottom
            bottom = man.get("body_bottom", rows[-1] if rows else body_top)
            k = max(1, round((bottom - body_top) / man["row_pitch"]))
            rows = list(np.linspace(body_top, bottom, k + 1)); src["rows"] = "manual pitch"
        else:
            src["rows"] = "manual range" if ("body_bottom" in man or "body_top" in man) else "auto"
        row_y = fill_missing_rows(rows)
    if len(row_y) < 2:
        raise ValueError("fewer than 2 row lines in the body; set row_y / body_top / body_bottom in MANUAL_CONFIG")

    page.grid = dict(col_x=col_x, col_names=names, row_y=[float(r) for r in row_y], body_top=float(row_y[0]),
                     body_bottom=float(row_y[-1]), table_bbox=list(page.table_bbox),
                     table_right=float(max(col_x[-1], x2)), source=src, notes=notes)
    return page.grid


def draw_grid(page):
    """Aligned page + white strip on top carrying the column names and the x of every column line."""
    g = page.grid
    strip = 150
    vis = cv2.copyMakeBorder(page.aligned, strip, 0, 0, 0, cv2.BORDER_CONSTANT, value=(255, 255, 255))
    top, bot = int(g["body_top"]) + strip, int(g["body_bottom"]) + strip
    for y in g["row_y"]:
        cv2.line(vis, (int(g["col_x"][0]), int(y) + strip), (int(g["col_x"][-1]), int(y) + strip), (0, 170, 0), 3)   # rows = green
    for i, x in enumerate(g["col_x"]):
        cv2.line(vis, (int(x), top - 60), (int(x), bot), (0, 0, 255), 4)                                               # columns = red
        cv2.putText(vis, str(int(x)), (int(x) - 35, 40 + (i % 2) * 34), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)
    for i, name in enumerate(g["col_names"]):
        cx = int((g["col_x"][i] + g["col_x"][i + 1]) / 2)
        cv2.putText(vis, name[:12], (max(cx - 50, 0), 115 + (i % 2) * 28), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (150, 0, 150), 2)
    cv2.line(vis, (int(g["col_x"][0]), int(g["body_top"]) + strip), (int(g["col_x"][-1]), int(g["body_top"]) + strip), (255, 0, 255), 6)  # header|body
    return vis


for pg in PAGES:
    if pg.table_bbox is None:
        continue
    try:
        g = resolve_grid(pg)
    except Exception as e:
        pg.grid = {}
        pg.warnings.append(str(e))
        print(f"{pg.uid}: GRID FAILED — {e}")
        continue
    pitch = float(np.median(np.diff(g["row_y"])))
    print(f"{pg.uid}: {len(g['col_names'])} columns ({g['source']['cols']}), {len(g['row_y']) - 1} body rows ({g['source']['rows']}), "
          f"median row pitch {pitch:.0f}px, body y {g['body_top']:.0f}..{g['body_bottom']:.0f}")
    print("   col_x =", [int(v) for v in g["col_x"]])
    print("   names =", g["col_names"])
    for note in g["notes"]:
        print("   note:", note)
    vis = draw_grid(pg)
    save_debug(pg, "04_rows_columns.png", vis)
    if viewed(pg):
        show(vis, f"DETECTED ROWS (green) & COLUMNS (red) — {pg.uid}", figsize=(9, 12))


In [ ]:
# ============================================================================
# STAGE 10 — ROWS  ->  RECORDS
# The printed grid rows are only ~65-85 px tall but handwriting wraps: "Pipe 12.70 x" is on one printed row and
# "1.5 x 442" on the next. So one handwritten RECORD may cover 1..max_rows_per_record printed rows.
#
#   * ink test  : pixels darker than INK_GRAY_THRESH, with the printed grid lines masked out
#   * a new record STARTS on a printed row whose ANCHOR cell (DATE / S.NO) contains ink (and the row has other handwriting)
#   * following rows without an anchor belong to that record (up to max_rows_per_record, trailing empty rows dropped)
#   * rows with ink but no anchor are reported (not silently merged) — e.g. the diagonal "strike-through" pen line
#     drawn over unused rows is ignored this way
# Set the template's anchor to None to get one record per inked printed row instead.
# ============================================================================
def grid_band_mask(page, half=6):
    """Mask of the printed grid (bands of +-half px around every row / column line) — used to ignore lines in ink tests."""
    g = page.grid
    m = np.zeros(page.gray.shape, np.uint8)
    xa, xb = int(g["col_x"][0]) - half, int(g["col_x"][-1]) + half
    ya, yb = int(g["row_y"][0]) - half, int(g["row_y"][-1]) + half
    for y in g["row_y"]:
        m[max(int(y) - half, 0): int(y) + half + 1, max(xa, 0): xb] = 1
    for x in g["col_x"]:
        m[max(ya, 0): yb, max(int(x) - half, 0): int(x) + half + 1] = 1
    return m


def cell_ink_matrix(page):
    """ink[r, c] = number of pen-ink pixels inside printed cell (row r, column c), grid lines excluded."""
    g = page.grid
    ink = ((page.gray < INK_GRAY_THRESH) & (grid_band_mask(page) == 0)).astype(np.uint8)
    ink = cv2.morphologyEx(ink, cv2.MORPH_OPEN, np.ones((2, 2), np.uint8))          # drop single-pixel dust
    integ = cv2.integral(ink)
    R, C = len(g["row_y"]) - 1, len(g["col_x"]) - 1
    out = np.zeros((R, C), int)
    for r in range(R):
        ya, yb = int(g["row_y"][r]) + 4, int(g["row_y"][r + 1]) - 4
        for c in range(C):
            xa, xb = int(g["col_x"][c]) + 4, int(g["col_x"][c + 1]) - 4
            out[r, c] = integ[yb, xb] - integ[ya, xb] - integ[yb, xa] + integ[ya, xa]
    return out


def group_records(page):
    g, T = page.grid, page.template
    man = get_manual(page)
    anchor = man.get("anchor", T.get("anchor"))
    max_rows = man.get("max_rows_per_record", T.get("max_rows_per_record", 2))
    ink = cell_ink_matrix(page)
    R = ink.shape[0]
    row_ink = ink.sum(axis=1)
    if anchor is None:
        starts = [r for r in range(R) if row_ink[r] >= MIN_INK_PIXELS]
        ends = [s + 1 for s in starts]
    else:
        a = g["col_names"].index(anchor)
        # a record starts where the anchor cell has ink AND the row has entry text in some other column
        # (a big circled S.NO spills into the next printed row: anchor ink only -> not a new record)
        starts = [r for r in range(R) if ink[r, a] >= MIN_ANCHOR_INK and row_ink[r] - ink[r, a] >= MIN_INK_PIXELS]
        ends = []
        for i, s in enumerate(starts):
            stop = min(starts[i + 1] if i + 1 < len(starts) else R, s + max_rows)
            while stop - 1 > s and row_ink[stop - 1] - ink[stop - 1, a] < MIN_INK_PIXELS:   # trim trailing rows with no entry text
                stop -= 1
            ends.append(stop)
    records = []
    for n, (s, e) in enumerate(zip(starts, ends), start=1):
        records.append(dict(rec_no=n, first_row=s, last_row=e - 1, y1=g["row_y"][s], y2=g["row_y"][e], cells={}))
    used = {r for rec in records for r in range(rec["first_row"], rec["last_row"] + 1)}
    a_ink = ink[:, g["col_names"].index(anchor)] if anchor else np.zeros(R, int)
    orphans = [r for r in range(R) if r not in used and row_ink[r] - a_ink[r] >= 3 * MIN_INK_PIXELS]
    if orphans:
        page.warnings.append(f"ink in printed rows {orphans} is not attached to any record "
                             f"(no {anchor!r} entry / beyond max_rows_per_record)")
    page.records = records
    page.detect["cell_ink"] = ink
    return records


for pg in PAGES:
    if not pg.grid:
        continue
    recs = group_records(pg)
    print(f"{pg.uid}: {len(recs)} record(s) -> printed rows " + ", ".join(f"#{r['rec_no']}: {r['first_row']}-{r['last_row']}" for r in recs))
    for w in pg.warnings:
        print("   warning:", w)


In [ ]:
# ============================================================================
# STAGE 11 — COLUMN CROPS WITH PADDING   (IMPORTANT R&D REQUIREMENT)
# Never cut exactly on the grid: handwriting crosses the printed lines.
#
#     padding = 10                       # pixels (the MINIMUM)
#     left  = x1 - padding
#     right = x2 + padding
#
# ...but 10 px is not enough when the entry is WIDER THAN ITS COLUMN, which happens constantly in the narrow QTY
# columns (~67 px at 300 DPI for a 4-digit number). The crop therefore grows outwards past the printed line for as
# long as the ink continues, and stops at the first white gutter (CROP_GUTTER_PX) - the gap that proves where this
# entry ends and the neighbour's begins. If no gutter is found within CROP_MAX_OVERFLOW the crop is NOT extended:
# packed columns cannot be separated safely, and importing the neighbour's digits is worse than clipping.
# A clipped digit is unreadable, and a half-digit is what sends the OCR model into a repetition loop.
#
# LAST column:   left  = x1 - padding
#                right = table_right + padding      (LAST_COL_MODE = "table")
#                right = right edge of the page      (LAST_COL_MODE = "page", default) — entries such as "Bending out"
#                often run past the last printed line, so the crop is extended completely to the edge.
# Vertically a record crop spans its printed rows +- V_PADDING. The FIRST data row is special: handwriting on it
# often rises above the header/body line, so the crop reaches up to V_PADDING_FIRST px above it, stopping just
# below any header text found in that gap (see first_row_headroom).
# Each crop is cut from the CLAHE-enhanced gray page and stored under its column HEADER NAME.
# ============================================================================
def record_ink_profile(page, rec):
    """(ink pixels per x inside this record's rows, mask of the printed column lines).

    The printed HORIZONTAL lines run the full width of the page, so their rows are dropped first — otherwise every
    column would look inked. The printed VERTICAL lines are not dropped but flagged: when the scan walks outwards
    they must be stepped over, since the line itself says nothing about where the handwriting ends.
    """
    g = page.grid
    H, W = page.gray.shape
    y1, y2 = max(int(rec["y1"]), 0), min(int(rec["y2"]), H)
    band = (page.gray[y1:y2] < INK_GRAY_THRESH).astype(np.uint8)
    for y in g["row_y"]:                                    # the rows where a printed line is expected
        yy = int(round(y)) - y1
        lo, hi = max(yy - LINE_HALF_PX, 0), yy + LINE_HALF_PX + 1
        if hi > 0 and lo < band.shape[0]:
            band[lo:hi] = 0
    # a scan is never perfectly straight, so a curved line can dodge the rows above. Anything still running
    # horizontally for a quarter of an inch is a printed line too, and one surviving line would ink every
    # column and hide every gutter.
    k = max(30, int(0.25 * DPI))
    band = cv2.subtract(band, cv2.morphologyEx(band, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (k, 1))))
    prof = band.sum(axis=0)

    is_line = np.zeros(W, bool)
    for x in g["col_x"]:
        xx = int(round(x))
        is_line[max(xx - LINE_HALF_PX, 0): xx + LINE_HALF_PX + 1] = True
    return prof, is_line


def column_overflow(prof, is_line, start, direction):
    """How far (px) the handwriting runs past a printed column line; 0 when it stops at the line.

    Walks outwards, skipping the printed line's own columns, until CROP_GUTTER_PX consecutive white columns are
    found: that white gap is the PROOF of where this entry ends and the neighbouring one begins.

    If no such gap appears within CROP_MAX_OVERFLOW the answer is 0, NOT the maximum. When the columns are packed
    edge to edge there is no way to tell this entry's ink from the neighbour's, and extending blindly pulls the
    neighbour's digits into the crop - on a real REJ QTY cell that produced a crop holding three columns at once
    ('400', '0', '2A'), which is worse than the clipping it was meant to fix. Without proof, stay conservative
    and keep the plain PADDING crop.
    """
    gap = reach = 0
    proved = False
    for step in range(1, CROP_MAX_OVERFLOW + 1):
        x = start + direction * step
        if not (0 <= x < len(prof)):
            break
        if is_line[x]:
            continue                                        # the printed line proves nothing either way
        if prof[x] > CROP_INK_EPS:
            reach, gap = step, 0
        else:
            gap += 1
            if gap >= CROP_GUTTER_PX:
                proved = True
                break
    return reach + CROP_EDGE_MARGIN if (proved and reach) else 0


def first_row_headroom(page, y1, left, right):
    """How far the FIRST data row's crop may reach above the header/body line, in px.

    A fixed 2 px used to chop the tops off letters written high on the first line. Now the gap above the line is
    inspected inside this column's own x-range: if it is blank, the full V_PADDING_FIRST is taken; if the printed
    header name is in the way, the crop stops 2 px below it. Never returns less than V_PADDING_TOP.
    """
    H, W = page.gray.shape
    y1 = int(round(y1))
    lo = max(y1 - V_PADDING_FIRST, 0)
    band = page.gray[lo:y1, max(int(left), 0): min(int(right), W)]
    if band.size == 0:
        return V_PADDING_TOP
    ink = (band < INK_GRAY_THRESH).astype(np.uint8)
    # the header/body line itself is ink too, so drop the long horizontal runs first: what is left is header TEXT
    k = max(15, min(int(LINE_KERNEL_FRAC * band.shape[1]), LINE_RUN_CAP_PX))
    runs = cv2.morphologyEx(ink, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (k, 1)))
    thick = cv2.morphologyEx(ink, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (1, LINE_HALF_PX + 2)))
    text = cv2.subtract(ink, cv2.subtract(runs, thick))      # a line is LONG and THIN; a shaded/bold header is not
    rows = np.flatnonzero(text.sum(axis=1) > 3)
    if rows.size == 0:
        return V_PADDING_FIRST                              # nothing but the line above -> take the whole buffer
    return int(np.clip(y1 - (lo + int(rows.max()) + 2), V_PADDING_TOP, V_PADDING_FIRST))


def crop_bounds(page, rec, c, prof=None, is_line=None):
    g = page.grid
    H, W = page.gray.shape
    x1, x2 = g["col_x"][c], g["col_x"][c + 1]
    if prof is None:
        prof, is_line = record_ink_profile(page, rec)
    left = x1 - max(PADDING, column_overflow(prof, is_line, int(round(x1)), -1))
    right = x2 + max(PADDING, column_overflow(prof, is_line, int(round(x2)), +1))
    if c == len(g["col_names"]) - 1:                                   # last column
        right = W if LAST_COL_MODE == "page" else g["table_right"] + PADDING
    top = rec["y1"] - (first_row_headroom(page, rec["y1"], left, right) if rec["first_row"] == 0 else V_PADDING)
    bottom = rec["y2"] + V_PADDING
    box = [max(int(round(left)), 0), max(int(round(top)), 0), min(int(round(right)), W), min(int(round(bottom)), H)]
    cell = [int(round(x1)), int(round(rec["y1"])), int(round(right if c == len(g["col_names"]) - 1 else x2)), int(round(rec["y2"]))]
    return box, cell


def make_crops(page):
    g = page.grid
    for rec in page.records:
        prof, is_line = record_ink_profile(page, rec)          # once per record, reused by all of its columns
        for c, name in enumerate(g["col_names"]):
            box, cell = crop_bounds(page, rec, c, prof, is_line)
            l, t, r, b = box
            # y of every printed row line INSIDE the crop: a wrapped entry continues below one of them, so
            # these are the split points the reader uses instead of guessing the text lines from the ink.
            splits = [int(round(y)) - t for y in g["row_y"] if t + 8 < y < b - 8]
            rec["cells"][name] = dict(col_index=c, crop_box=box, cell_box=cell, row_splits=splits,
                                      crop=page.gray[t:b, l:r].copy())


def draw_crop_boxes(page, max_records=None):
    """Aligned page with every padded crop rectangle (colours alternate per column; last column reaches the page edge)."""
    vis = page.aligned.copy()
    palette = [(255, 0, 0), (0, 140, 255), (0, 170, 0), (200, 0, 200)]
    for rec in page.records[:max_records]:
        for name, cell in rec["cells"].items():
            l, t, r, b = cell["crop_box"]
            col = palette[cell["col_index"] % len(palette)]
            cv2.rectangle(vis, (l, t), (r, b), col, 3)
            x1, y1, x2, y2 = cell["cell_box"]
            cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 0, 0), 1)             # thin black = the un-padded printed cell
    return vis


for pg in PAGES:
    if not pg.grid:
        continue
    make_crops(pg)
    n = sum(len(r["cells"]) for r in pg.records)
    print(f"{pg.uid}: {n} crops ({len(pg.records)} records x {len(pg.grid['col_names'])} columns)")
    for r in pg.records[:1]:
        for name, cell in list(r["cells"].items())[:2] + list(r["cells"].items())[-1:]:
            print(f"   record 1 / {name:22s} cell x {cell['cell_box'][0]}..{cell['cell_box'][2]}  ->  crop x {cell['crop_box'][0]}..{cell['crop_box'][2]}")


In [ ]:
# ============================================================================
# STAGE 12 — REMOVE TABLE / GRID LINES FROM EVERY CROP  (without eating the handwriting)
# We already know where the printed lines are (Stage 9), so the removal is *guided*:
#   1. for every known row / column line that falls inside the crop, look +-LINE_GUIDE_TOL px around its expected
#      position and snap to the darkest ridge (scans are slightly curved, so the line is rarely exactly where predicted)
#   2. mark a band of +-LINE_HALF_PX px around the snapped position as the area that MAY be erased
#   3. inside that band erase only the pixels that really belong to a printed line, i.e. that are part of a
#      horizontal / vertical run long enough that no pen stroke could produce it
#   4. PROTECT the crossings only: a pen stroke that CROSSES a printed line has ink on both sides of it, so the
#      erase mask keeps those columns (a few px of line stub survive there — harmless for OCR, unlike a hole
#      punched through the letter). Ink that merely sits ON the line, like a word written along it, has ink on one
#      side only and does NOT protect it, so the line under the text is still erased. Protecting any nearby ink
#      instead (the first attempt) left the line completely intact under every word: measured on a real crop, the
#      printed line was 559 px wide and 565 px of it came out protected.
#   5. cv2.inpaint fills what was erased from the surrounding paper.
# Erasing the whole band (the earlier behaviour) destroyed every descender and loop that touched a line.
# LINE_REMOVAL_MODE = "morph" instead detects the long runs anywhere in the crop (no grid knowledge).
# ============================================================================
def _snap(profile, pos):
    """Position of the darkest ridge within +-LINE_GUIDE_TOL of `pos` (falls back to pos if the region is blank)."""
    lo, hi = max(int(round(pos)) - LINE_GUIDE_TOL, 0), min(int(round(pos)) + LINE_GUIDE_TOL + 1, len(profile))
    if hi - lo < 3 or profile[lo:hi].max() < 8:
        return pos
    return lo + int(np.argmax(profile[lo:hi]))


def _line_runs(ink, w, h):
    """Masks of the horizontal / vertical runs that are long enough to be a printed grid line, never a pen stroke."""
    kh = max(15, min(int(LINE_KERNEL_FRAC * w), LINE_RUN_CAP_PX))
    kv = max(15, min(int(LINE_KERNEL_FRAC * h), LINE_RUN_CAP_PX))
    hl = cv2.morphologyEx(ink, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (kh, 1)))
    vl = cv2.morphologyEx(ink, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (1, kv)))
    return hl, vl


def crossings(hand):
    """Mask of the places where a pen stroke passes THROUGH a printed line.

    A stroke crossing a horizontal line has ink above it and below it; one crossing a vertical line has ink to its
    left and to its right. Ink on one side only (a word resting on the line) is not a crossing. `hand` is the ink
    that is not part of a long run, i.e. the handwriting.
    """
    n = LINE_CROSS_PX
    kv, kh = np.ones((n + 1, 1), np.uint8), np.ones((1, n + 1), np.uint8)
    above = cv2.dilate(hand, kv, anchor=(0, n))        # ink at or above this pixel
    below = cv2.dilate(hand, kv, anchor=(0, 0))        # ink at or below
    left = cv2.dilate(hand, kh, anchor=(n, 0))
    right = cv2.dilate(hand, kh, anchor=(0, 0))
    return cv2.bitwise_or(cv2.bitwise_and(above, below), cv2.bitwise_and(left, right))


def remove_grid_lines(gray, box, page):
    """Return (line-removed gray crop, erased mask). `box` = crop position on the page."""
    l, t, r, b = box
    h, w = gray.shape
    ink = cv2.morphologyEx(
        cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 31, 10),
        cv2.MORPH_OPEN, np.ones((2, 2), np.uint8))                 # ink = 255, single-pixel dust dropped
    hl, vl = _line_runs(ink, w, h)
    lines = cv2.bitwise_or(hl, vl)

    if LINE_REMOVAL_MODE == "morph":
        mask = lines
    else:
        dark = 255.0 - gray.astype(np.float32)
        row_prof, col_prof = dark.mean(axis=1), dark.mean(axis=0)
        band = np.zeros_like(gray)                                 # where a printed line is ALLOWED to be
        for y in page.grid["row_y"]:
            yy = y - t
            if -LINE_HALF_PX <= yy < h + LINE_HALF_PX:
                yy = _snap(row_prof, yy)
                band[max(int(yy) - LINE_HALF_PX, 0): int(yy) + LINE_HALF_PX + 1, :] = 255
        for x in page.grid["col_x"]:
            xx = x - l
            if -LINE_HALF_PX <= xx < w + LINE_HALF_PX:
                xx = _snap(col_prof, xx)
                band[:, max(int(xx) - LINE_HALF_PX, 0): int(xx) + LINE_HALF_PX + 1] = 255
        mask = cv2.bitwise_and(lines, band)                        # real line pixels, at a known line position

    mask = cv2.dilate(mask, np.ones((3, 3), np.uint8))             # catch the soft edge of the printed line
    mask = cv2.subtract(mask, crossings(cv2.subtract(ink, lines)))  # keep where a stroke passes through
    clean = cv2.inpaint(gray, mask, LINE_INPAINT_RAD, cv2.INPAINT_TELEA)
    return clean, mask


def clear_margins(clean, crop_box, cell_box, sides=("left", "right")):
    """Blank the ink that has come in from the NEIGHBOURING column.

    The padding deliberately reaches past the printed cell, which drags in the tail of the neighbour's entry: on a
    real page the PROBLEM DESCRIPTION crop opened with '00', the end of the next column's 2400, and the cell was
    read as 'CHI b'. Three things have to be true together before a blob is treated as the neighbour's, because
    each one alone gets it wrong:
      * it RUNS OFF the outer edge of the crop  - an entry of our own is whole inside the crop;
      * it is separated from every other blob by a white gutter - in a narrow QTY column our own digits touch both
        crop edges, and on the edge test alone they were wiped (LOT QTY lost 98.7% of its ink);
      * it stays within the outer eighth of the cell - our own first digit can be cut off by the crop and stand
        apart from the rest of the number, but it does not sit right against the border.
    """
    l, t, r, b = crop_box
    x1, _, x2, _ = cell_box
    h, w = clean.shape
    ink = (clean < INK_GRAY_THRESH).astype(np.uint8)
    k = max(15, min(int(LINE_KERNEL_FRAC * w), LINE_RUN_CAP_PX))       # printed lines would connect everything
    ink = cv2.subtract(ink, cv2.morphologyEx(ink, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (k, 1))))
    ink = cv2.subtract(ink, cv2.morphologyEx(ink, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (1, k))))
    ink = cv2.dilate(ink, np.ones((3, 3), np.uint8))                   # rejoin strokes broken by the line removal
    n, lab, stats, _ = cv2.connectedComponentsWithStats(ink)
    if n < 2:
        return clean

    cx1, cx2 = int(np.clip(x1 - l, 0, w)), int(np.clip(x2 - l, 0, w))
    keep_out = max(int((cx2 - cx1) / 8), CROP_GUTTER_PX)               # the outer eighth of the printed cell
    out = clean.copy()
    for side in sides:
        for i in set(np.unique(lab[:, 0 if side == "left" else w - 1])) - {0}:
            x, y, cw, ch, area = stats[i]
            if area > 0.4 * h * w:                                     # never wipe the whole crop
                continue
            other = ((lab > 0) & (lab != i)).sum(axis=0)
            if side == "left":
                if x >= cx1 - CROP_EDGE_MARGIN or x + cw > cx1 + keep_out:
                    continue
                gutter = other[x + cw: x + cw + CROP_GUTTER_PX]
            else:
                if x + cw <= cx2 + CROP_EDGE_MARGIN or x < cx2 - keep_out:
                    continue
                gutter = other[max(x - CROP_GUTTER_PX, 0): x]
            if len(gutter) == CROP_GUTTER_PX and (gutter <= CROP_INK_EPS).all():
                out[lab == i] = 255
    return out


def clean_crops(page):
    for rec in page.records:
        last = len(page.grid["col_names"]) - 1
        for name, cell in rec["cells"].items():
            cell["clean"], cell["line_mask"] = remove_grid_lines(cell["crop"], cell["crop_box"], page)
            # the last column is extended to the page edge on purpose, so its right margin is left alone
            sides = ("left",) if (cell["col_index"] == last and LAST_COL_MODE == "page") else ("left", "right")
            cell["clean"] = clear_margins(cell["clean"], cell["crop_box"], cell["cell_box"], sides)
            # ink inside the un-padded printed cell only (padding may contain a neighbour's stroke)
            l, t, r, b = cell["crop_box"]
            x1, y1, x2, y2 = cell["cell_box"]
            core = cell["clean"][max(y1 - t, 0): y2 - t, max(x1 - l, 0): x2 - l]
            cell["ink_pixels"] = int((core < INK_GRAY_THRESH).sum())
            cell["empty"] = cell["ink_pixels"] < MIN_INK_PIXELS


for pg in PAGES:
    if not pg.grid:
        continue
    clean_crops(pg)
    cells = [c for r in pg.records for c in r["cells"].values()]
    print(f"{pg.uid}: line-removed {len(cells)} crops; {sum(not c['empty'] for c in cells)} contain handwriting, {sum(c['empty'] for c in cells)} empty")


In [ ]:
# ============================================================================
# STAGE 13 — DEBUG VIEWS + SAVING THE CROPS
# Shows / saves, for every page:  original page (Stage 5) | detected table (Stage 7) | rows & columns (Stage 9) |
# padded crop boxes | individual column crops | line-removed crops.  Crops are also written one-by-one to
# OUTPUT_DIR/crops/<page>/ as  r01_<COLUMN NAME>.png  and  r01_<COLUMN NAME>_clean.png.
# ============================================================================
try:
    _FONT = ImageFont.load_default(size=14)
except TypeError:                                   # older Pillow
    _FONT = ImageFont.load_default()


def make_sheet(page, kinds=("crop",), scale=0.5, empty_note=True):
    """Contact sheet: one block per record, one tile per column (header name on top), tiles keep their real relative widths.
    kinds = ("crop",) raw padded crops | ("clean",) line-removed | ("crop", "clean") raw row above cleaned row."""
    names = page.grid["col_names"]
    tiles = {}
    colw = [0] * len(names)
    for rec in page.records:
        for kind in kinds:
            for c, name in enumerate(names):
                img = rec["cells"][name][kind]
                tile = cv2.resize(img, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
                tiles[(rec["rec_no"], kind, c)] = tile
                colw[c] = max(colw[c], tile.shape[1])
    gap, head = 4, 22
    xs = np.cumsum([0] + [w + gap for w in colw])
    rows_h = {(rec["rec_no"], kind): max(tiles[(rec["rec_no"], kind, c)].shape[0] for c in range(len(names)))
              for rec in page.records for kind in kinds}
    H = head + sum(h + gap for h in rows_h.values()) + gap
    canvas = np.full((H, int(xs[-1]), 3), 200, np.uint8)
    y = head
    for rec in page.records:
        for kind in kinds:
            for c in range(len(names)):
                t = to_bgr(tiles[(rec["rec_no"], kind, c)])
                canvas[y: y + t.shape[0], xs[c]: xs[c] + t.shape[1]] = t
            y += rows_h[(rec["rec_no"], kind)] + gap
    pil = Image.fromarray(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
    d = ImageDraw.Draw(pil)
    for c, name in enumerate(names):
        d.text((xs[c] + 2, 3), name[:max(3, colw[c] // 8)], fill=(120, 0, 120), font=_FONT)
    return cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)


def show_sheet(sheet, title, max_width_in=15):
    h, w = sheet.shape[:2]
    show(sheet, title, figsize=(max_width_in, max(2.0, max_width_in * h / w + 0.6)))


def save_crops(page):
    if not SAVE_DEBUG:
        return
    base = os.path.join(OUTPUT_DIR, "crops", page.uid)
    os.makedirs(base, exist_ok=True)
    for rec in page.records:
        for name, cell in rec["cells"].items():
            stem = f"r{rec['rec_no']:02d}_{safe_name(name)}"
            cv2.imwrite(os.path.join(base, stem + ".png"), cell["crop"])
            cv2.imwrite(os.path.join(base, stem + "_clean.png"), cell["clean"])


for pg in PAGES:
    if not pg.grid:
        continue
    save_crops(pg)
    boxes = draw_crop_boxes(pg)
    sheet_raw = make_sheet(pg, ("crop",))
    sheet_clean = make_sheet(pg, ("crop", "clean"))
    save_debug(pg, "05_crop_boxes.png", boxes)
    save_debug(pg, "06_crops_raw.png", sheet_raw)
    save_debug(pg, "07_crops_raw_vs_line_removed.png", sheet_clean)
    if not viewed(pg):
        continue
    x1, y1, x2, y2 = pg.table_bbox
    top = max(int(pg.records[0]["y1"]) - 120, 0) if pg.records else y1
    bot = min(int(pg.records[-1]["y2"]) + 120, boxes.shape[0]) if pg.records else y2
    show(boxes[top:bot], f"PADDED CROP BOXES (coloured) vs printed cells (thin black) — {pg.uid}", figsize=(14, 5))
    show_sheet(sheet_raw, f"INDIVIDUAL COLUMN CROPS (padded) — {pg.uid}")
    show_sheet(sheet_clean, f"LINE-REMOVED CROPS — {pg.uid}   (each record: raw row on top, line-removed row below)")


In [ ]:
# ============================================================================
# SUMMARY — one row per (page, record, column): where its crop is and whether it holds handwriting.
# This is the spatial map you will feed to the OCR stage later: every crop is keyed by page -> record -> column NAME.
# ============================================================================
rows = []
for pg in PAGES:
    for rec in pg.records:
        for name, cell in rec["cells"].items():
            rows.append(dict(pdf=pg.pdf, template=pg.template_key, record=rec["rec_no"], printed_rows=f"{rec['first_row']}-{rec['last_row']}",
                             column=name, crop_box=cell["crop_box"], has_handwriting=not cell["empty"], ink_pixels=cell["ink_pixels"]))
summary = pd.DataFrame(rows)
print(f"{len(summary)} crops, {int(summary['has_handwriting'].sum()) if len(summary) else 0} with handwriting")
for pg in PAGES:
    for w in pg.warnings:
        print(f"WARNING {pg.uid}: {w}")
if COPY_OUTPUT_TO_DRIVE:                                    # optional: keep results on Drive (Colab)
    shutil.copytree(OUTPUT_DIR, COPY_OUTPUT_TO_DRIVE, dirs_exist_ok=True)
    print("copied output to", COPY_OUTPUT_TO_DRIVE)
summary.head(30)


## OCR on the cell crops — GOT-OCR-2.0

Stages 14–16 read every non-empty cell crop from Stage 11/12 with the pretrained **GOT-OCR-2.0** model
(no fine-tuning) and store one reading per cell in a single JSON file.

| # | Stage | What it does |
|---|-------|--------------|
| 14 | Load model | `stepfun-ai/GOT-OCR-2.0-hf`, OCR settings |
| 15 | Cell reader | ink-trim each crop; GOT reads the whole cell as one image (wrapped text included) |
| 16 | Run + save | every page → record → column, written to `ocr_output/ocr_cell_results.json` |

Empty cells (Stage 12 ink test) are skipped. Confidence is GOT's mean per-token generation probability.


In [ ]:
# ============================================================================
# STAGE 14 — LOAD THE OCR MODEL  (GOT-OCR-2.0, pretrained, no fine-tuning)
# Pretrained weights straight from the Hub. Needs a GPU for reasonable speed (a Colab T4 is plenty).
# ============================================================================
for pip_name, module in [("accelerate", "accelerate"), ("sentencepiece", "sentencepiece")]:
    ensure(pip_name, module)

# ---- which crops go to the model -----------------------------------------------
OCR_CROP_KIND  = "clean"        # "clean" = grid lines removed (Stage 12) | "crop" = raw padded crop (Stage 11)
OCR_SKIP_EMPTY = True           # do not send cells that the Stage 12 ink test marked empty
OCR_JSON_PATH  = os.path.join(OUTPUT_DIR, "ocr_cell_results.json")

# ---- model ---------------------------------------------------------------------
# GOT_MODEL_ID lives in Stage 1, because Stage 7b may already have needed the model to identify a form.
GOT_MAX_NEW_TOKENS = 128        # a cell holds a few words, not a page (the page-level notebook used 512).
                                # This is now only the CEILING: the budget per cell is scaled to the size of the
                                # crop (see token_budget), because room to ramble is what lets the model fall
                                # into a repetition loop on a crop it cannot read.
GOT_MIN_NEW_TOKENS = 12         # ... but never below this, so a short entry is never truncated
GOT_NO_REPEAT_NGRAM = 6         # block verbatim repeats of 6 tokens: handwriting in one cell never contains one,
                                # a degenerate 't t t t t t' loop always does. 0 turns the guard off.
LINE_SPLIT_SNAP = 30            # px: how far the split point may move off the printed row line to land in the
                                # real gap between the two lines of handwriting
GOT_LINE_SPLIT = True           # read a WRAPPED cell one text line at a time. Handed a two-line block, GOT returns
                                # only one of the lines: 'Pipe 12.70 x / 1.5 x 442' came back as '1.5 X 442' and
                                # 'Length variation 491 / to 50 percent 498+-1' as '498+-1'. False = one call per cell.

ensure_got()              # defined in Stage 7b; a no-op if that stage already loaded the model

print(f"model ready on {device} | crops used: {OCR_CROP_KIND!r} | output: {OCR_JSON_PATH}")


In [ ]:
# ============================================================================
# STAGE 15 — CELL READER
# The crops are already grayscale + CLAHE-enhanced (Stage 6) and 300 DPI, so no upscale / second CLAHE here (the page-level
# comparison notebook needed them; a cell crop does not). Per cell:
#   1. trim the crop to its ink bounding box + white margin  (drops the padding, the empty part of the wide last column)
#   2. GOT-OCR-2.0 reads the trimmed cell as ONE image — no line splitting needed: it reads a wrapped
#      entry (text spilling onto the 2nd printed row) in a single pass.
#   3. a WRAPPED entry (text carried onto the 2nd printed row) is read one text line at a time and the readings
#      are joined. Handed the whole block GOT returns only one of the lines, which was losing most of the content
#      of every multi-line cell on these forms.
#   4. two guards against the failure mode of a VLM on an unreadable crop, where it emits the same token until it
#      runs out of budget ('T 2 A d t t t t t ...'):
#        * the generation budget is scaled to the crop, so a 67 px wide QTY box cannot produce 128 tokens
#        * what comes back is checked for a repetition loop and suppressed if it is one. This check is NOT
#          redundant with the confidence number: a loop repeats high-probability tokens, so it comes back with a
#          HIGH mean confidence (0.88 on a real garbage reading) and no confidence threshold can catch it.
# ============================================================================
def _num(x):
    """JSON-safe number: NaN / inf / None -> None."""
    return None if x is None or not math.isfinite(x) else round(float(x), 4)


def ink_mask(gray):
    """Binary mask of dark ink on light paper (Otsu threshold)."""
    _, mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    return mask


def trim_to_ink(gray, margin_ratio=0.15, min_ink_pixels=10):
    """Trim to the bounding box of the ink, then add a white margin. None if (almost) no ink."""
    if gray.size == 0:
        return None
    mask = cv2.morphologyEx(ink_mask(gray), cv2.MORPH_OPEN, np.ones((2, 2), np.uint8))
    if np.count_nonzero(mask) < min_ink_pixels:
        return None
    ys, xs = np.nonzero(mask)
    trimmed = gray[ys.min(): ys.max() + 1, xs.min(): xs.max() + 1]
    m = int(min(trimmed.shape) * margin_ratio) + 4
    return cv2.copyMakeBorder(trimmed, m, m, m, m, cv2.BORDER_CONSTANT, value=255)


def token_budget(img):
    """Generation budget for one crop: about one token per 6 px of width, per line of text, clamped to
    [GOT_MIN_NEW_TOKENS, GOT_MAX_NEW_TOKENS]. A narrow QTY cell then gets ~28 tokens instead of 128."""
    h, w = img.shape[:2]
    lines = max(1, int(round(h / 70.0)))                      # a printed row is ~70 px at 300 DPI
    return int(np.clip(w // 6 * lines, GOT_MIN_NEW_TOKENS, GOT_MAX_NEW_TOKENS))


def is_degenerate(text):
    """True for the repetition loop a VLM produces on a crop it cannot read ('T 2 A d t t t t t ...').

    Checked on the decoded text rather than on the confidence, because such a loop repeats tokens the model is
    very sure about and therefore scores HIGH confidence — no threshold on the confidence can separate it.
    """
    toks = text.split()
    if len(toks) >= 8 and len(set(toks[-8:])) <= 2:            # the tail is one or two tokens over and over
        return True
    squashed = re.sub(r"\s+", "", text)
    if len(squashed) >= 12:
        top = max(squashed.count(ch) for ch in set(squashed))
        if top / len(squashed) > 0.6:                          # one character makes up most of a long reading
            return True
    return False


def split_into_lines(gray, min_line_ratio=0.25, min_gap_ratio=0.4):
    """Split a cell into its text lines, top to bottom; [gray] when there is only one.

    The printed grid lines are suppressed first: one surviving line would otherwise be counted as a line of text
    and sent to the model on its own.
    """
    h, w = gray.shape
    ink = (gray < INK_GRAY_THRESH).astype(np.uint8)
    k = max(15, min(int(LINE_KERNEL_FRAC * w), LINE_RUN_CAP_PX))
    ink = cv2.subtract(ink, cv2.morphologyEx(ink, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (k, 1))))
    profile = ink.sum(axis=1)
    if profile.max() == 0:
        return [gray]
    has_ink = profile > max(2, profile.max() * 0.10)
    runs, start = [], None
    for y, inked in enumerate(has_ink):
        if inked and start is None:
            start = y
        elif not inked and start is not None:
            runs.append([start, y])
            start = None
    if start is not None:
        runs.append([start, len(has_ink)])
    if not runs:
        return [gray]
    min_gap = max(2, int(np.median([b - a for a, b in runs]) * min_gap_ratio))
    merged = [runs[0]]
    for run in runs[1:]:
        if run[0] - merged[-1][1] < min_gap:
            merged[-1][1] = run[1]
        else:
            merged.append(run)
    tallest = max(b - a for a, b in merged)
    merged = [r for r in merged if (r[1] - r[0]) >= tallest * min_line_ratio]
    if len(merged) <= 1:
        return [gray]
    pad = max(2, min_gap // 2)
    return [gray[max(0, a - pad): min(h, b + pad)] for a, b in merged]


def text_line_cuts(prof, min_peak_frac=0.30, min_sep=15, valley_frac=0.75):
    """y positions where a cell's handwriting separates into text lines.

    Each line of writing is a hump in the row-ink profile, so the cuts are the valleys BETWEEN the humps. Two
    simpler rules both failed on real crops: thresholding the profile never separates 'Pipe 12.70 x' from
    '1.5 x 442' (their descenders and ascenders overlap, so the valley only dips to 26% of the peak and the
    profile stays above any threshold), and snapping to the quietest row near the printed line lands in the
    blank paper below the entry instead of in the gap between the lines.
    """
    p = np.convolve(prof.astype(float), np.ones(5) / 5.0, mode="same")
    if p.max() <= 0:
        return []
    peaks = []
    for y in range(1, len(p) - 1):
        if p[y] >= p.max() * min_peak_frac and p[y] >= p[y - 1] and p[y] > p[y + 1]:
            if peaks and y - peaks[-1] < min_sep:
                if p[y] > p[peaks[-1]]:
                    peaks[-1] = y                          # same hump, keep the taller row
            else:
                peaks.append(y)
    cuts = []
    for hi_a, hi_b in zip(peaks, peaks[1:]):
        v = hi_a + int(np.argmin(p[hi_a:hi_b]))
        if p[v] < valley_frac * min(p[hi_a], p[hi_b]):      # a real gap, not a wobble in one line
            cuts.append(v)
    return cuts


def split_cell(gray, row_splits=None):
    """The text lines of one cell crop, top to bottom; [gray] when there is only one.

    `row_splits` (the printed row boundaries inside the crop, measured in Stage 11) is used only to cap how many
    lines are possible - a record spanning two printed rows can hold two lines of writing. Where the split falls
    comes from the ink, because the writer does not follow the ruling: on a real PART NAME cell both lines sat
    above the printed line, which would have sliced the second one in half.
    """
    h, w = gray.shape
    ink = (gray < INK_GRAY_THRESH).astype(np.uint8)
    k = max(15, min(int(LINE_KERNEL_FRAC * w), LINE_RUN_CAP_PX))
    ink = cv2.subtract(ink, cv2.morphologyEx(ink, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (k, 1))))
    cuts = text_line_cuts(ink.sum(axis=1))
    # one printed line is a few px thick and arrives as several neighbouring values; cluster them, and ignore any
    # that sits against the top or bottom of the crop, where there is no room for a line of writing below it
    inner = sorted({int(y) for y in (row_splits or []) if 20 < int(y) < h - 20})
    clusters = [y for i, y in enumerate(inner) if i == 0 or y - inner[i - 1] > 8]
    max_lines = len(clusters) + 1 if row_splits is not None else 3
    if not cuts or max_lines < 2:
        return [gray]
    cuts = sorted(cuts)[:max_lines - 1]
    bands, prev = [], 0
    for y in cuts + [h]:
        if y - prev >= 12:                                  # ignore slivers
            bands.append(gray[prev:y])
        prev = y
    return bands or [gray]


def got_read_cell(gray, row_splits=None):
    """GOT-OCR-2.0 on one cell crop -> dict(text, confidence [, lines, error]).

    A wrapped entry is read one line at a time and the readings joined with a space; the individual lines are
    kept in `lines`. Confidence is the mean over the lines that produced text.
    """
    parts = split_cell(gray, row_splits) if GOT_LINE_SPLIT else [gray]
    reads = []
    for part in parts:
        part = trim_to_ink(part)
        # a band holding only a scrap - the stub of a printed line, a speck from the next column - must not be
        # sent to the model, or its guess gets appended to a perfectly good reading
        if part is not None and int((part < INK_GRAY_THRESH).sum()) >= MIN_INK_PIXELS // 3:
            reads.append(got_read_image(part))
    if not reads:
        return dict(text="", confidence=None)
    if len(reads) == 1:
        return reads[0]
    texts = [r["text"] for r in reads if r["text"]]
    confs = [r["confidence"] for r in reads if r["confidence"] is not None]
    out = dict(text=" ".join(texts), confidence=_num(np.mean(confs)) if confs else None,
               lines=[r["text"] for r in reads])
    errors = sorted({r["error"] for r in reads if r.get("error")})
    if errors:
        out["error"] = "; ".join(errors)
    return out


def got_read_image(trimmed):
    """One already-trimmed image -> dict(text, confidence). Confidence = mean per-token generation probability."""
    text, conf = got_generate(trimmed, token_budget(trimmed), GOT_NO_REPEAT_NGRAM)   # got_generate is in Stage 7b
    if is_degenerate(text):                                    # do not let a loop reach the CSV as a confident reading
        return dict(text="", confidence=None, error="degenerate repetition suppressed", raw=text[:80])
    return dict(text=text, confidence=_num(conf))


In [ ]:
# ============================================================================
# STAGE 16 — RUN GOT-OCR-2.0 ON EVERY CELL CROP  ->  ocr_cell_results.json
# For each page -> record -> column: OCR the crop with GOT-OCR-2.0 and store the reading.
# Empty cells (Stage 12 ink test) are not sent to the model when OCR_SKIP_EMPTY is True.
#
# JSON layout (OCR_JSON_PATH):
#   meta  : model, crop kind, timestamp
#   pages : { "<page uid>": { pdf, page, template, columns,
#             records: [ { record, printed_rows,
#                          got   : {COLUMN: text},          <- whole-record view, one entry per column
#                          cells : {COLUMN: { crop_file, crop_box, ink_pixels, empty,
#                                             got  : {text, confidence, error?} } } } ] } }
# crop_file is relative to OUTPUT_DIR (the PNGs written by Stage 13 when SAVE_DEBUG is True).
# confidence: mean per-token generation probability (null if unavailable).
# A cell whose reading was a repetition loop comes back as text "" with an `error` and the raw output in `raw`,
# so it shows up as blank for review instead of as a high-confidence wrong value.
# ============================================================================
def _run_model(fn, *args):
    """Call the model; a failure on one cell must not stop the whole run."""
    try:
        return fn(*args)
    except Exception as e:
        return dict(text="", confidence=None, error=f"{type(e).__name__}: {e}")


def run_ocr(pages):
    out = {}
    for pg in pages:
        if not pg.grid or not pg.records:
            continue
        names = pg.grid["col_names"]
        suffix = "_clean" if OCR_CROP_KIND == "clean" else ""
        page_out = dict(pdf=pg.pdf, page=pg.page_no, template=pg.template_key, columns=names, records=[])
        for rec in pg.records:
            cells_out, got_view = {}, {}
            for name in names:
                cell = rec["cells"][name]
                stem = f"r{rec['rec_no']:02d}_{safe_name(name)}"
                entry = dict(crop_file=f"crops/{pg.uid}/{stem}{suffix}.png", crop_box=cell["crop_box"],
                             ink_pixels=cell["ink_pixels"], empty=bool(cell["empty"]))
                if cell["empty"] and OCR_SKIP_EMPTY:
                    entry["got"] = None
                    got_view[name] = ""
                else:
                    entry["got"] = _run_model(got_read_cell, cell[OCR_CROP_KIND], cell.get("row_splits"))
                    got_view[name] = entry["got"]["text"]
                cells_out[name] = entry
            page_out["records"].append(dict(record=rec["rec_no"], printed_rows=f"{rec['first_row']}-{rec['last_row']}",
                                            got=got_view, cells=cells_out))
            print(f"{pg.uid}: record {rec['rec_no']} done")
        out[pg.uid] = page_out
    return out


t0 = time.time()
OCR_RESULTS = run_ocr(PAGES)
payload = dict(meta=dict(got_model=GOT_MODEL_ID, crop_kind=OCR_CROP_KIND,
                         skip_empty=OCR_SKIP_EMPTY, created=datetime.now().isoformat(timespec="seconds")),
               pages=OCR_RESULTS)
with open(OCR_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)
print(f"\nSaved {OCR_JSON_PATH}  ({time.time() - t0:.0f}s)")

# quick look: one row per OCR'd cell
flat = [dict(page=uid, record=r["record"], column=col, got=c["got"]["text"], got_conf=c["got"]["confidence"])
        for uid, p in OCR_RESULTS.items() for r in p["records"] for col, c in r["cells"].items() if c["got"] is not None]
ocr_df = pd.DataFrame(flat)
if len(ocr_df):
    print(f"{len(ocr_df)} cells OCR\'d | mean GOT confidence {ocr_df['got_conf'].mean():.3f} | "
          f"{int((ocr_df['got'].str.strip() == '').sum())} came back empty")
ocr_df


In [ ]:
# ============================================================================
# STAGE 17 — STRUCTURED CSV  (ONE CSV PER PDF; one row per RECORD, one column per FORM COLUMN)
# ocr_cell_results.json is cell-level: one entry per crop. This stage pivots it back into the shape of the
# printed form, so the result can be opened in Excel or loaded into a database row by row.
#
# One CSV per source PDF  ->  OUTPUT_DIR/structured/<pdf name>.csv
# (a PDF always has exactly one template — Stage 5 picks it from the file name — so all of its pages and
#  records share the same columns; a multi-page PDF stays in ONE file, its pages told apart by the `page` column).
# Layout:  source_pdf | template | page | record | printed_rows | <FORM COLUMNS ...> | <COLUMN> (conf) ... | mean_conf | low_conf
#   mean_conf : mean GOT confidence over the non-empty cells of that record
#   low_conf  : names of the columns read below LOW_CONF_THRESH
#   implausible: names of the columns whose reading is the wrong SHAPE for that column (a QTY that is not a
#                number, a DATE that is not a date) — a much better review signal than the confidence
# The JSON FILE is the input (not the in-memory OCR_RESULTS), so this cell can be re-run on an old result
# without repeating the OCR.
# ============================================================================
# A reading that does not have the shape its column requires is a far better review signal than the confidence
# number: on this data a garbage cell scored 0.88 while a correct one scored 0.41. "QTY" columns must be a plain
# number, DATE must look like a date; anything else is listed in `implausible`.
COLUMN_PATTERNS = [
    (r"QTY",  r"^\d{1,7}$",                                             "not a number"),
    # a date may come back with its separators lost - '9/6/26' reads as '916126' - so a bare run of digits is
    # accepted too; what gets flagged is a date carrying letters, like 'Y 8126'.
    (r"^DATE$", r"^\d{1,2}\s*[/\-. ]?\s*\d{1,2}\s*[/\-. ]?\s*\d{2,4}$", "not a date"),
]

STRUCTURED_DIR  = os.path.join(OUTPUT_DIR, "structured")
INCLUDE_CONF    = True      # also write a "<COLUMN> (conf)" column next to every form column
LOW_CONF_THRESH = 0.60      # readings below this are listed in `low_conf`
META_COLS       = ["source_pdf", "template", "page", "record", "printed_rows"]


def implausible_columns(row, columns):
    """Names of the columns whose reading does not match the shape their header requires (blanks are not flagged)."""
    bad = []
    for name in columns:
        value = str(row.get(name) or "").strip()
        if not value:
            continue
        for col_pat, value_pat, _why in COLUMN_PATTERNS:
            if re.search(col_pat, name, re.I) and not re.match(value_pat, value):
                bad.append(name)
                break
    return bad


def clean_text(t):
    """One cell of GOT output -> one CSV value: collapse whitespace, drop the markdown fences it sometimes adds."""
    return re.sub(r"\s+", " ", str(t or "")).strip().strip("`").strip()


def structured_tables(json_path=None, payload=None):
    """Read ocr_cell_results.json -> {pdf stem: DataFrame}, one row per record, one table per source PDF."""
    if payload is None:
        with open(json_path or OCR_JSON_PATH, encoding="utf-8") as f:
            payload = json.load(f)

    tables = {}
    for uid, page in payload["pages"].items():
        stem = os.path.splitext(page["pdf"])[0]                # every page of one PDF -> one table
        entry = tables.setdefault(stem, dict(columns=[], rows=[]))
        for name in page["columns"]:                           # union, first-seen order (header_names may override)
            if name not in entry["columns"]:
                entry["columns"].append(name)
        for rec in page["records"]:
            row = dict(source_pdf=page["pdf"], template=page["template"], page=page["page"],
                       record=rec["record"], printed_rows=rec["printed_rows"])
            confs, low = [], []
            for name in page["columns"]:
                got = (rec["cells"].get(name) or {}).get("got") or {}      # empty / skipped cell -> {}
                text = clean_text(got.get("text"))
                conf = got.get("confidence")
                row[name] = text
                if INCLUDE_CONF:
                    row[f"{name} (conf)"] = conf
                if text and conf is not None:                              # blanks must not drag the mean down
                    confs.append(conf)
                    if conf < LOW_CONF_THRESH:
                        low.append(name)
            row["mean_conf"] = round(float(np.mean(confs)), 4) if confs else None
            row["low_conf"] = "; ".join(low)
            row["implausible"] = "; ".join(implausible_columns(row, page["columns"]))
            entry["rows"].append(row)

    out = {}
    for stem, e in sorted(tables.items()):
        order = META_COLS + e["columns"] \
                + ([f"{n} (conf)" for n in e["columns"]] if INCLUDE_CONF else []) \
                + ["mean_conf", "low_conf", "implausible"]
        df = pd.DataFrame(e["rows"]).reindex(columns=order)
        out[stem] = df.sort_values(["page", "record"]).reset_index(drop=True)
    return out


os.makedirs(STRUCTURED_DIR, exist_ok=True)
STRUCTURED = structured_tables()
for stem, df in STRUCTURED.items():
    path = os.path.join(STRUCTURED_DIR, f"{safe_name(stem)}.csv")
    df.to_csv(path, index=False, encoding="utf-8-sig")     # utf-8-sig so Excel shows the text correctly
    form_cols = [c for c in df.columns if c not in META_COLS and not c.endswith("(conf)")
                 and c not in ("mean_conf", "low_conf", "implausible")]
    print(f"{path}\n    {len(df)} records x {len(form_cols)} form columns | "
          f"{int((df[form_cols] != '').to_numpy().sum())} cells filled | "
          f"{int(df['low_conf'].astype(bool).sum())} records with a low-confidence cell | "
          f"{int(df['implausible'].astype(bool).sum())} with an implausible value")
if not STRUCTURED:
    print("nothing to write — no pages in", OCR_JSON_PATH)

# preview the first PDF without the confidence columns
if STRUCTURED:
    first = next(iter(STRUCTURED))
    print(f"\npreview — {first}")
    display(STRUCTURED[first][[c for c in STRUCTURED[first].columns if not c.endswith("(conf)")]].head(20))


In [ ]:
# ============================================================================
# STAGE 18 - SAVE ocr_df + DOWNLOAD THE RESULTS TO YOUR PC
# Colab cannot write to your computer's disk directly; the browser download is the way. So everything goes in ONE zip
# (one download = no "allow multiple downloads" prompt). The zip contains, at its top level:
#     ocr_cell_results.json   ocr_cell_results.csv (= ocr_df)   structured/   crops/   debug/
# It lands in your browser's Downloads folder (or asks where to save if Chrome's "Ask where to save each file" is on).
# The results themselves are already permanent: OUTPUT_DIR is on Drive (Colab) or a local folder;
# the zip is only a convenience copy, so on Colab it is built on the fast local disk, not on Drive.
# ============================================================================
OCR_CSV_PATH = os.path.join(OUTPUT_DIR, "ocr_cell_results.csv")
ocr_df.to_csv(OCR_CSV_PATH, index=False, encoding="utf-8-sig")         # utf-8-sig so Excel shows the text correctly
print("saved", OCR_CSV_PATH)

# archive is created OUTSIDE OUTPUT_DIR so it does not try to include itself
ZIP_DIR = "/content" if IN_COLAB else os.path.dirname(os.path.abspath(OUTPUT_DIR))
zip_path = shutil.make_archive(os.path.join(ZIP_DIR, "ocr_output"), "zip", OUTPUT_DIR)
print(f"zipped {OUTPUT_DIR} -> {zip_path} ({os.path.getsize(zip_path) / 1e6:.1f} MB)")

if IN_COLAB:
    from google.colab import files
    files.download(zip_path)
    print("Results also kept permanently in:", OUTPUT_DIR)
else:
    print("Not running in Colab - zip is here:", os.path.abspath(zip_path))